# Error Analysis: Identify characteristics of misclassified sentences

- For the sentences that were incorrectly classified, what are the characteristics in them?
- For the ones that are classified as a prediction, do our prediction properties exist?

In [7]:
import os
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [8]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

## Load Data

In [9]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
# combine_data_path = os.path.join(base_data_path, 'financial_phrase_bank/combined_generated_fin_phrase_bank')
train_data = os.path.join(base_data_path, 'classification_results/train_synthetic-v1_2026-03-23/seed3')

In [10]:
# model_results_path = os.path.join(combine_data_path, 'inference_chronicle2050_2026-03-07_21-51-47.csv')
test_data = os.path.join(train_data, 'external_fpb-maya-binary-imbalanced-96d-v1/', 'ml_classifiers_fpb-maya-binary-imbalanced-96d-v1.csv')

model_results_df = DataProcessing.load_from_file(test_data, 'csv', sep=',')
cols_to_drop = ["Base Sentence Embedding", "Dataset Name", "Author Type", "maya_label"]
compare_y_vs_yhats_df = DataProcessing.drop_df_columns(model_results_df, cols_to_drop)
compare_y_vs_yhats_df

,Base Sentence,Sentence Label,perceptron,sgd_classifier,logistic_regression,ridge_classifier,decision_tree_classifier,random_forest_classifier,gradient_boosting_classifier,support_vector_machine_classifier,x_gradient_boosting_classifier
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,1,1,1,0,1,1,1,1
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,1,1,0,0,1,1,1,0,1
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,1,1,1,1,1,1,1,1,1
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,0,0,0,0,1,0,0,0,0
4,The company also estimates the already carried out investments to lead to an increase in its net sales for 2010 from 2009 when they reached EUR 141.7 million .,1,1,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
4841,LONDON MarketWatch -- Share prices ended lower in London Monday as a rebound in bank stocks failed to offset broader weakness for the FTSE 100 .,0,0,0,0,0,0,0,1,0,0
4842,"Rinkuskiai 's beer sales fell by 6.5 per cent to 4.16 million litres , while Kauno Alus ' beer sales jumped by 6.9 per cent to 2.48 million litres .",0,0,0,0,0,1,0,0,0,0
4843,"Operating profit fell to EUR 35.4 mn from EUR 68.8 mn in 2007 , including vessel sales gain of EUR 12.3 mn .",0,0,0,0,0,0,0,0,0,0
4844,"Net sales of the Paper segment decreased to EUR 221.6 mn in the second quarter of 2009 from EUR 241.1 mn in the second quarter of 2008 , while operating profit excluding non-recurring items rose to EUR 8.0 mn from EUR 7.6 mn .",0,0,0,0,0,0,0,0,0,0


In [11]:
model_col_names = compare_y_vs_yhats_df.columns.to_list()[2:]
model_col_names, len(model_col_names)

(['perceptron',
  'sgd_classifier',
  'logistic_regression',
  'ridge_classifier',
  'decision_tree_classifier',
  'random_forest_classifier',
  'gradient_boosting_classifier',
  'support_vector_machine_classifier',
  'x_gradient_boosting_classifier'],
 9)

In [32]:
compare_y_vs_mv_df = compare_y_vs_yhats_df[model_col_names].mode(axis=1)
compare_y_vs_yhats_df['Majority Vote'] = compare_y_vs_mv_df
compare_y_vs_yhats_df

,Base Sentence,Sentence Label,perceptron,sgd_classifier,logistic_regression,ridge_classifier,decision_tree_classifier,random_forest_classifier,gradient_boosting_classifier,support_vector_machine_classifier,x_gradient_boosting_classifier,Majority Vote
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,1,1,1,0,1,1,1,1,1
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,1,1,0,0,1,1,1,0,1,1
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,1,1,1,1,1,1,1,1,1,1
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,0,0,0,0,1,0,0,0,0,0
4,The company also estimates the already carried out investments to lead to an increase in its net sales for 2010 from 2009 when they reached EUR 141.7 million .,1,1,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4841,LONDON MarketWatch -- Share prices ended lower in London Monday as a rebound in bank stocks failed to offset broader weakness for the FTSE 100 .,0,0,0,0,0,0,0,1,0,0,0
4842,"Rinkuskiai 's beer sales fell by 6.5 per cent to 4.16 million litres , while Kauno Alus ' beer sales jumped by 6.9 per cent to 2.48 million litres .",0,0,0,0,0,1,0,0,0,0,0
4843,"Operating profit fell to EUR 35.4 mn from EUR 68.8 mn in 2007 , including vessel sales gain of EUR 12.3 mn .",0,0,0,0,0,0,0,0,0,0,0
4844,"Net sales of the Paper segment decreased to EUR 221.6 mn in the second quarter of 2009 from EUR 241.1 mn in the second quarter of 2008 , while operating profit excluding non-recurring items rose to EUR 8.0 mn from EUR 7.6 mn .",0,0,0,0,0,0,0,0,0,0,0


## Get misalignment 

- Per y and $y_{hat}$
- Per model
- Possibly realign where all models disagree on misalignment. Or, maybe not as worse case could be they misalign?

In [ ]:
def get_misclassified_sentences(df, model_col_names):
    tps, fns, tns, fps = [], [], [], []

    for idx, row in df.iterrows():
        sentence = row['Base Sentence']
        y = row['Sentence Label']

        for model_col_name in model_col_names:
            y_hat = row[model_col_name]

            # y = 1 (positive class)
            if y == 1:
                if y_hat == 1:  # TP
                    tps.append((idx, sentence, y, y_hat, model_col_name))
                else:  # FN
                    fn = (idx, sentence, y, y_hat, model_col_name)
                    fns.append(fn)
                    if idx < 3:
                        print(f"{idx}-{sentence}")
                        print(f"\tMismatch: {fn}")

            # y = 0 (negative class)
            elif y == 0:
                if y_hat == 0:  # TN
                    tns.append((idx, sentence, y, y_hat, model_col_name))
                else:  # FP
                    fp = (idx, sentence, y, y_hat, model_col_name)
                    fps.append(fp)
                    if idx < 3:
                        print(f"{idx}-{sentence}")
                        print(f"\tMismatch: {fp}")

    # Build outputs
    col_names = ["Row", "Sentence", "True Label", "Model Label", "Model Name"]
    tps_df = pd.DataFrame(tps, columns=col_names)
    tns_df = pd.DataFrame(tns, columns=col_names)
    fns_df = pd.DataFrame(fns, columns=col_names)
    fps_df = pd.DataFrame(fps, columns=col_names)

    return tps_df, tns_df, fns_df, fps_df

In [ ]:
tps_df, tns_df, fns_df, fps_df = get_misclassified_sentences(compare_y_vs_yhats_df, model_col_names)


0-With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
	Mismatch: (0, 'With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .', 1, 0, 'decision_tree_classifier')
1-According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .
	Mismatch: (1, "According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .", 1, 0, 'logistic_regression')
1-According to the company 's updated strategy for the years 2009-2012 , Ba

,Row,Sentence,True Label,Model Label,Model Name
0,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,perceptron
1,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,sgd_classifier
2,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,logistic_regression
3,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,ridge_classifier
4,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,random_forest_classifier
...,...,...,...,...,...
3274,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,sgd_classifier
3275,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,logistic_regression
3276,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,ridge_classifier
3277,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,decision_tree_classifier


| $ y $ | $ y_{hat} $ | Outcome |
|----------------|----------------|----------|
| 1              | 1              | **TP**   |
| 1              | 0              | **FN**  (miss out) |
| 0              | 0              | **TN**   |
| 0              | 1              | **FP**  (inflate) |

**Confusion Matrix (sklearn-style layout)**

| y \ y_hat |   0   |   1   |
|-----------|-------|-------|
| **0**     |  TN   |  FP  (inflate) |
| **1**     |  FN  (miss out) |  TP   |

In [25]:
len(tps_df), len(tns_df), len(fns_df), len(fps_df)

(3279, 22712, 843, 16780)

In [24]:
tps_df

,Row,Sentence,True Label,Model Label,Model Name
0,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,perceptron
1,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,sgd_classifier
2,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,logistic_regression
3,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,ridge_classifier
4,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,1,random_forest_classifier
...,...,...,...,...,...
3274,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,sgd_classifier
3275,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,logistic_regression
3276,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,ridge_classifier
3277,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,1,decision_tree_classifier


In [21]:
tns_df

,Row,Sentence,True Label,Model Label,Model Name
0,459,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .",0,0,random_forest_classifier
1,460,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",0,0,perceptron
2,460,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",0,0,sgd_classifier
3,460,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",0,0,logistic_regression
4,460,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",0,0,ridge_classifier
...,...,...,...,...,...
22707,4845,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",0,0,decision_tree_classifier
22708,4845,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",0,0,random_forest_classifier
22709,4845,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",0,0,gradient_boosting_classifier
22710,4845,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",0,0,support_vector_machine_classifier


In [22]:
fns_df

,Row,Sentence,True Label,Model Label,Model Name
0,0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,0,decision_tree_classifier
1,1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,0,logistic_regression
2,1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,0,ridge_classifier
3,1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,0,support_vector_machine_classifier
4,3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,0,perceptron
...,...,...,...,...,...
838,456,"Operating profit excluding non-recurring items decreased to EUR 6.2 mn from EUR 16.8 mn in 2007 , representing 2.3 % of net sales .",1,0,support_vector_machine_classifier
839,456,"Operating profit excluding non-recurring items decreased to EUR 6.2 mn from EUR 16.8 mn in 2007 , representing 2.3 % of net sales .",1,0,x_gradient_boosting_classifier
840,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,0,random_forest_classifier
841,457,Repeats sees 2008 operating profit down y-y ( Reporting by Helsinki Newsroom ) Keywords : TECNOMEN-RESULTS,1,0,gradient_boosting_classifier


In [23]:
fps_df

,Row,Sentence,True Label,Model Label,Model Name
0,458,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",0,1,perceptron
1,458,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",0,1,sgd_classifier
2,458,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",0,1,logistic_regression
3,458,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",0,1,ridge_classifier
4,458,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .",0,1,decision_tree_classifier
...,...,...,...,...,...
16775,4834,Hobby Hall 's sales decrease 26 pct due to implementing a new information system that involved changing in the principal of posting sales .,0,1,gradient_boosting_classifier
16776,4834,Hobby Hall 's sales decrease 26 pct due to implementing a new information system that involved changing in the principal of posting sales .,0,1,support_vector_machine_classifier
16777,4834,Hobby Hall 's sales decrease 26 pct due to implementing a new information system that involved changing in the principal of posting sales .,0,1,x_gradient_boosting_classifier
16778,4841,LONDON MarketWatch -- Share prices ended lower in London Monday as a rebound in bank stocks failed to offset broader weakness for the FTSE 100 .,0,1,gradient_boosting_classifier


In [ ]:
false_negative = mislabelled_pivot_table.loc[1]   # rows whose True Label == 1; given prediction, model classified as non-prediction
false_negative

In [ ]:


# 2️⃣  (Optional) reset the index if you prefer flat columns
false_positive = false_positive.reset_index()
true1_table = true1_table.reset_index()

# 3️⃣  Quick checks
print(f"Rows where True Label 0: {len(false_positive)}")
print(f"Rows where True Label 1: {len(true1_table)}")

# Now you can plot or analyse each table separately, e.g.:
#   plt.bar(true0_table['Model Name'], true0_table['Model Label'])

In [ ]:
def groupby_model_name(df, model_name):
    filt_llama = (df['Model Name'] == model_name)
    filt_df = df[filt_llama]
    return filt_df, len(filt_df)

model_dfs = []
for idx, row in misclassified_sentences_df.iterrows():
    model_name = row['Model Name']
    model_df, model_df_length = groupby_model_name(misclassified_sentences_df, model_name)
    # print(model_df)
    model_df["N Misclassified"] = model_df_length

    model_dfs.append(model_df)

In [ ]:
each_model_df = pd.concat(model_dfs)
each_model_df.head(3)

In [ ]:
len(model_results_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1️⃣  Count the number of mismatches (i.e. rows) per model
# ------------------------------------------------------------------
miscounts = misclassified_sentences_df['Model Name'].value_counts()
misclassified_ratio = (miscounts / len(model_results_df))
properly_classified_ratio = 1 - misclassified_ratio # Should match classification
total_classified_ratio = misclassified_ratio + properly_classified_ratio
# print(f"{misclassified_ratio}, {properly_classified_ratio} = {total_classified_ratio}")
# ------------------------------------------------------------------
# 3.  Pretty‑print: one row per model, percentages to 2 decimals
# ------------------------------------------------------------------
ratios_df = pd.DataFrame({
    'Model': miscounts.index,
    'Mis‑classified': misclassified_ratio,
    'Correctly classified': properly_classified_ratio,
    'Total classified': total_classified_ratio,
    'N': len(model_results_df),
})

print(ratios_df.to_string(index=False,
                          float_format=lambda x: f'{x:.2%}'))
# ------------------------------------------------------------------
# 2️⃣  Plot
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# bar chart: one bar per model
miscounts.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='black')

# aesthetics
ax.set_xlabel('Model')
ax.set_ylabel('Number of Misclassifications')
ax.set_title('Misclassifications per Model')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# show exact count on top of each bar
for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------
# 1️⃣  Prepare the data ------------------------------------------------
# -------------------------------------------------------------
miscounts                = misclassified_sentences_df['Model Name'].value_counts()
misclassified_ratio      = miscounts / len(model_results_df)        # proportion that were wrong
properly_classified_ratio= 1 - misclassified_ratio                # proportion that were right
total_classified_ratio   = misclassified_ratio + properly_classified_ratio  # = 1

# DataFrame ready for printing / plotting
ratios_df = pd.DataFrame(
    {
        'Model': miscounts.index,
        'Mis‑classified': misclassified_ratio,
        'Correctly classified': properly_classified_ratio,
        'Total classified': total_classified_ratio,
        'N': len(model_results_df),
    }
)

print(ratios_df.to_string(index=False, float_format=lambda v: f'{v:.2%}'))

# -------------------------------------------------------------
# 2️⃣  Plot the stacked bar ------------------------------------
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

# ── split the bars into two stacks ----------------------------
ax.bar(
    ratios_df.index,
    ratios_df['Mis‑classified'],
    width=0.8,
    color='#EB2D2D',
    label='Mis‑classified'
)

ax.bar(
    ratios_df.index,
    ratios_df['Correctly classified'],
    width=0.8,
    bottom=ratios_df['Mis‑classified'],
    color='#2EBC9B',
    label='Correctly classified'
)

# ── cosmetics --------------------------------------------------
ax.set_xlabel('Model')
ax.set_ylabel('Proportion')
ax.set_title('Mis‑classification vs Correct classification per model')
ax.set_xticklabels(ratios_df['Model'], rotation=45, ha='right')
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend()

# ── annotate bars with percentage values -----------------------
for i, (mis, cor) in enumerate(zip(ratios_df['Mis‑classified'],
                                    ratios_df['Correctly classified'])):
    ax.text(i, mis / 2, f'{mis:.1%}', ha='center', va='center', color='white', fontsize=8)
    ax.text(i, mis + cor / 2,
            f'{cor:.1%}',
            ha='center',
            va='center',
            color='white',
            fontsize=8)

plt.tight_layout()
plt.show()